# Experiment 16 - Confidence calibration across failure-reasoning types

**Objective.** Measure how closely a model's stated confidence matches its empirical
accuracy, and how that relationship varies across *reasoning type*, *amount of context*,
and *question difficulty*, on the failure-reasoning capabilities an LLM-based
failure-recovery pipeline depends on.

These are REFLECT *failure* episodes, so the questions are built around the divergence
between what the plan intended and what was actually observed, and are anchored on the
dataset ground truth (`gt_failure_step`, `gt_failure_reason`, `specified_missing_steps`).
The model under test is kept out of generation. Each question has four content options
plus a fifth *"an option not listed here"*, option order is randomised, and an option's
score is the model's next-token probability for its letter (after KnowNo, Ren et al. 2023).
A generation-time validity gate enforces that every item is well-formed, free of gimme
distractors, and context-necessary (its answer cannot be read off the stem and the weak
panel cannot answer it without the context). Difficulty is labelled by an independent
five-LLM panel. Calibration is summarised with accuracy, ECE, reliability curves and mean
confidence, with two-level (task to episode) bootstrap confidence intervals.

**Reasoning types** - T1 outcome verification (what was actually observed after an action
whose intended effect did not occur), T2 failure localization (which step first went
wrong), T3 failure attribution (why the task failed), T4 missing-step identification.
**Context regimes** - C1 full trace, C2 local window, C3 plan only.
The grid keeps the **7 valid (type, regime) cells**: T1 in C1/C2, T2 in C1, T3 in C1/C2,
T4 in C2/C3.

A counterfactual regime and a recovery type were prototyped and dropped after audit: the
counterfactual clause never changed the answer and contradicted the trace, and a sound
recovery action cannot be derived from the available ground truth (only the failure is
labelled, not the fix).

## 1. Configuration

In [ ]:
# Pin BLAS/OpenMP threads to 1 before importing the pipeline; the open3d/numpy stack
# otherwise spins on this machine. Must precede the imports below.
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import re, json, math, pickle, random, hashlib, time
from dataclasses import dataclass, field
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from scipy.stats import spearmanr

from reflect.core.data import load_episode_data, load_task_details
from reflect.pipelines.fast_validation import scene_graphs_mem, summary_mem
from reflect.core.constants import NAME_MAP
from reflect.llm.prompter import LocalLLMPrompter, PortkeyLLMPrompter



from reflect.core.paths import sim_data_root, analysis_experiment_dir

DATA_ROOT = sim_data_root()
OUT_DIR   = analysis_experiment_dir("llm_calibration_target_distribution")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Every (task, episode) in the local sim dataset.
EPISODES = sorted(
    (task_dir.name, ep_dir.name)
    for task_dir in DATA_ROOT.iterdir() if task_dir.is_dir() and not task_dir.name.startswith(".")
    for ep_dir in task_dir.iterdir() if ep_dir.is_dir() and not ep_dir.name.startswith(".")
)

# Five-LLM difficulty panel (none of them a model under test), answering with no reasoning.
PANEL_MODELS = ["mistral:7b", "llama3.2:3b", "granite3.3:8b", "phi4-mini:3.8b", "command-r7b"]
OLLAMA_URL = "http://localhost:11434"

# Models under test; calibration is computed and compared for each.
MUT_MODELS = ["qwen3.5:9b", "qwen3.5:27b", "gpt-5.4", "qwen3.6:27b", "qwen3.6:35b"]
MODEL_BACKENDS = {
    "qwen3.5:9b":  {"backend": "local"},
    "qwen3.5:27b": {"backend": "local"},
    "gpt-5.4":     {"backend": "portkey", "reasoning_effort": "none", "virtual_key_env": "PORTKEY_VIRTUAL_KEY"},
    "qwen3.6:27b": {"backend": "local"},
    "qwen3.6:35b": {"backend": "local"},
}

SEED         = 16      # option shuffling (reproducible questions)
CAP_PER_CELL = 3       # max questions per (type, regime) per episode
ECE_BINS     = 10      # equal-width confidence bins for ECE / reliability
N_BOOT       = 2000    # two-level bootstrap resamples
WITH_AUDIO   = 1       # 0 -> read task.json 'sounds' (skip the heavy audio model)

# Difficulty bucket from panel solvability (number of correct panel answers, 0..5).
def difficulty_bucket(solvability):
    if solvability <= 1: return "hard"
    if solvability <= 3: return "medium"
    return "easy"

print("panel:", PANEL_MODELS)
print("episodes:", len(EPISODES))

## 2. Episode layers

Each episode is processed by the REFLECT perception pipeline into a hierarchy of layers:
the **plan** (intended action sequence), the **goal** layer (the executed action per
keyframe) and the **observation** layer (object states, spatial relations and auditory
cues per keyframe). We parse the L2 (subgoal-level) captions into structured `Step`
records. Scene-graph generation is slow, so parsed episodes are cached to disk.

In [ ]:
# Caption parsing: turn each L2 caption into a Step (time, goal, states, relations).
REL_VERBS = ["is inside", "is on top of", "is on the right of", "is on the left of",
             "is in front of", "is behind", "is under"]

@dataclass
class Step:
    time: str; action: str; states: dict; relations: list; auditory: str = ""; raw: str = ""

def _sentences(text):
    return [s.strip() for s in text.split(". ") if s.strip()]

def parse_relation(sent):
    sent = sent.rstrip(".")
    for rv in REL_VERBS:
        if f" {rv} " in f" {sent} ":
            subj, obj = sent.split(f" {rv} ", 1)
            return (subj.strip(), rv, obj.strip())
    return None

def parse_caption(cap):
    cap = cap.strip()
    time = cap.split(".", 1)[0]
    m = re.search(r"(?:Action|Goal):\s*(.*?)\.\s*Visual observation:", cap)
    action = m.group(1).strip() if m else ""
    am = re.search(r"Auditory observation:\s*(.*?)\.?\s*$", cap)
    auditory = am.group(1).strip().rstrip(".") if am else ""
    vm = re.search(r"Visual observation:\s*(.*?)(?:\s*Auditory observation:|$)", cap, re.S)
    visual = (vm.group(1).strip() if vm else "").rstrip("\n").strip()
    states, relations = {}, []
    for i, sent in enumerate(_sentences(visual)):
        sent = sent.rstrip(".")
        rel = parse_relation(sent)
        if rel is not None:
            relations.append(rel)
        elif i == 0 and " is " not in sent:
            for item in sent.split(", "):
                sm = re.match(r"^(.*?)\s*\((.*)\)\s*$", item.strip())
                if sm: states[sm.group(1).strip()] = sm.group(2).strip()
    return Step(time, action, states, relations, auditory, cap)

@dataclass
class Episode:
    task: str; name: str; plan: list; success: str; steps: list; objs: list
    gt_failure_step: object = None
    gt_failure_reason: str = ""
    specified_missing_steps: list = field(default_factory=list)

def _load_episode_raw(task, name):
    path = str(DATA_ROOT / task / name)
    td = load_task_details(path + "/task.json")
    events, td, object_list, interact, nav = load_episode_data(path, td)
    local_sgs, global_sg, key_frames = scene_graphs_mem(events, object_list, nav, interact, WITH_AUDIO, {}, td)
    _, _, _, l2_captions = summary_mem(events, nav, interact, WITH_AUDIO, {}, td, local_sgs, key_frames)
    steps = [parse_caption(c) for c in l2_captions]
    return Episode(task, name, td["actions"], td["success_condition"], steps, object_list,
                   td.get("gt_failure_step"), td.get("gt_failure_reason", "") or "",
                   td.get("specified_missing_steps") or [])

def load_episodes(force=False):
    # episodes_v2.pkl carries the REFLECT ground-truth failure fields, built by the pipeline
    # (force=True) or migrated from the old episodes.pkl by the helper script.
    cache = OUT_DIR / "episodes_v2.pkl"
    if cache.exists() and not force:
        return pickle.loads(cache.read_bytes())
    eps = [_load_episode_raw(task, name) for task, name in EPISODES]
    cache.write_bytes(pickle.dumps(eps))
    return eps

episodes = load_episodes(force=False)
print(f"loaded {len(episodes)} episodes")
print(f"  with gt_failure_step: {sum(1 for e in episodes if e.gt_failure_step)} | "
      f"with specified_missing_steps: {sum(1 for e in episodes if e.specified_missing_steps)}")

## 3. Question engine

Templates build reproducible items with known answers from the parsed trace, the plan, and
the REFLECT ground truth. Each item has one reasoning type and one context regime. The
design makes the context load-bearing: for T1 the trap distractor is the textbook intended
effect (which is wrong, because the step diverged), so a context-free guesser fails; T2 and
T3 must read the trace to localize or explain the failure; T4 must compare the reduced plan
to the goal. Distractors are plausible and type-correct (a fact from the wrong timestep, an
inverted relation, a present-but-not-missing step, the correct action on the wrong object,
another failure mode), never physically impossible. Every item gets four content options
plus a fifth *"an option not listed here"*; the five are shuffled with a per-item seed and
assigned letters A-E.

The context regime controls how much trace is shown: **C1 full trace** gives every step,
**C2 local window** gives only the steps around the queried or failure step, **C3 plan only**
gives the plan and goal with no observations.

In [ ]:
# object / action language
_NUM = {"1":"first","2":"second","3":"third","4":"fourth","5":"fifth","6":"sixth"}

def obj_text(obj):
    if obj in NAME_MAP: return NAME_MAP[obj]
    if "-" in obj:
        base, num = obj.rsplit("-", 1)
        base_t = NAME_MAP.get(base, re.sub(r"(?<!^)(?=[A-Z])", " ", base).lower())
        return f"{_NUM.get(num, num)} {base_t}"
    return re.sub(r"(?<!^)(?=[A-Z])", " ", obj).lower()

_VERB = {"navigate_to_obj":("move to",1),"pick_up":("pick up",1),"put_in":("put",2,"in"),
         "put_on":("put",2,"on"),"toggle_on":("toggle on",1),"toggle_off":("toggle off",1),
         "open_obj":("open",1),"close_obj":("close",1),"slice_obj":("slice",1),
         "crack_obj":("crack",1),"pour":("pour",2,"into")}

def action_text(action):
    parts = action.strip("()").split(", "); verb, args = parts[0], parts[1:]
    spec = _VERB.get(verb)
    if spec is None: return action
    if spec[1] == 1: return f"{spec[0]} the {obj_text(args[0])}"
    return f"{spec[0]} the {obj_text(args[0])} {spec[2]} the {obj_text(args[1])}"

def parse_action(action):
    parts = action.strip("()").split(", "); return parts[0], parts[1:]

def is_interact(action):
    return parse_action(action)[0] != "navigate_to_obj"

# parse an executed L2 goal string into (verb_phrase, [object phrases])
def parse_goal(goal):
    g = goal.strip(); low = g.lower()
    if low.startswith("pour liquid from "):
        body = g[len("pour liquid from "):]
        parts = re.split(r"\s+into\s+", body, maxsplit=1)
        return ("pour", [p.strip().lower() for p in parts])
    for pre, verb in [("toggle on ","toggle on"),("toggle off ","toggle off"),
                      ("pick up ","pick up"),("open ","open"),("close ","close"),
                      ("slice ","slice"),("crack ","crack")]:
        if low.startswith(pre):
            return (verb, [g[len(pre):].strip().lower()])
    if low.startswith("put "):
        body = g[len("put "):]
        parts = re.split(r"\s+(in|on)\s+", body, maxsplit=1)
        if len(parts) == 3:
            return ("put " + parts[1], [parts[0].strip().lower(), parts[2].strip().lower()])
        return ("put", [body.strip().lower()])
    return (None, [g.lower()])

def goal_phrase(goal):
    """Normalise an executed goal to the same 'the'-style as action_text."""
    verb, objs = parse_goal(goal)
    if verb is None: return goal.lower()
    if verb in ("put in","put on"):
        return f"put the {objs[0]} {verb.split()[1]} the {objs[1]}"
    if verb == "pour":
        return f"pour the {objs[0]} into the {objs[1]}"
    return f"{verb} the {objs[0]}"

# affordance pools
PICKABLE_POOL  = ["bowl","plate","mug","cup","pan","kettle","knife","apple","tomato","potato","egg","pot","bread"]
TOGGLABLE_POOL = ["faucet","toaster","microwave","coffee machine","stove burner","floor lamp","television"]
OPENABLE_POOL  = ["fridge","microwave","cabinet","drawer","laptop","box"]
SLICEABLE_POOL = ["apple","bread","lettuce","potato","tomato"]
_ORDINALS = ["first ","second ","third ","fourth ","fifth ","sixth "]
def base_obj(phrase):
    p = phrase
    for w in _ORDINALS: p = p.replace(w, "")
    return p.strip()

def _pool_for(verb):
    if verb in ("toggle on","toggle off"): return TOGGLABLE_POOL
    if verb in ("open","close"):           return OPENABLE_POOL
    if verb == "slice":                    return SLICEABLE_POOL
    return PICKABLE_POOL          # pick up / put in / put on / pour / crack

def scene_objs(ep):
    """Object phrases mentioned anywhere in the episode (goals, states, relations)."""
    objs = set()
    for st in ep.steps:
        v, os_ = parse_goal(st.action)
        objs.update(os_)
        objs.update(st.states.keys())
        for r in st.relations:
            if r[0] != "nothing": objs.add(r[0])
            objs.add(r[2])
    objs.discard("robot gripper")
    return sorted(o for o in objs if o)   # sorted: deterministic across processes (str hash is randomized)

def alt_object(ep, verb, correct_obj, dest, seed):
    """A plausible, type-appropriate object different from the correct one (and dest)."""
    pool = _pool_for(verb)
    excl = {base_obj(correct_obj)} | ({base_obj(dest)} if dest else set())
    in_scene = [o for o in scene_objs(ep) if base_obj(o) in pool and base_obj(o) not in excl]
    extra    = [o for o in pool if o not in excl]
    rng = random.Random(seed)
    rng.shuffle(in_scene); rng.shuffle(extra)
    cands = in_scene + extra
    return cands[0] if cands else None

# rendering
def fmt_state(obj, st): return f"the {obj} is {st}"
def fmt_rel(r):         return f"the {r[0]} {r[1]} the {r[2]}"

def render_obs(step):
    parts = [fmt_state(o, s) for o, s in step.states.items()]
    parts += [fmt_rel(r) for r in step.relations if r[0] != "nothing"]
    if not parts: parts = ["nothing notable is observed"]
    return ", ".join(parts)

def goal_line(step): return f"At {step.time}, the robot's goal was to {step.action.lower()}."
def obs_line(step):  return f"At {step.time}, the observed state was: {render_obs(step)}."

def plan_text(ep, steps=None):
    steps = ep.plan if steps is None else steps
    return "; ".join(f"{k+1}) {action_text(a)}" for k, a in enumerate(steps))

_REL_RE = re.compile(r"^the (.+?) (is inside|is on top of|is on the right of|is on the left of|is in front of|is behind|is under) the (.+)$")
def _parse_fact(fact):
    if fact.startswith("nothing is inside the robot gripper"):
        return ("rel", ("nothing", "is inside", "robot gripper"))
    m = _REL_RE.match(fact)
    if m: return ("rel", (m.group(1), m.group(2), m.group(3)))
    m = re.match(r"^the (.+?) is (.+)$", fact)
    if m: return ("state", (m.group(1), m.group(2)))
    return (None, None)

def _state_entailed(state_str, attr):
    if attr == "broken": return "broken" in state_str
    if attr.startswith("not "): return attr in state_str
    if attr == "sliced":  return "sliced" in state_str and "not sliced" not in state_str
    if attr == "cracked": return "cracked" in state_str and "not cracked" not in state_str
    if attr in ("open", "closed", "turned on", "turned off", "empty"):
        return attr in state_str
    return attr in state_str   # compound / full attr -> substring

def entails(step, fact):
    """True if a fact phrase holds at the step (entailment, not exact string match)."""
    kind, val = _parse_fact(fact)
    if kind == "rel":
        a, rel, b = val
        if a == "nothing":
            return any(r[0] == "nothing" for r in step.relations)
        return (a, rel, b) in [tuple(r) for r in step.relations]
    if kind == "state":
        o, attr = val
        return _state_entailed(step.states.get(o, ""), attr)
    return False

# failure-centric helpers (REFLECT GT)
def _secs(t):
    try:
        mm, ss = str(t).split(":"); return int(mm) * 60 + int(ss)
    except Exception:
        return None

def _norm_failtime(gfs):
    while isinstance(gfs, list):
        if not gfs: return None
        gfs = gfs[0]
    return gfs if isinstance(gfs, str) else None

def failure_step_index(ep):
    """Index into ep.steps of the (first) ground-truth failure step, matched by time."""
    ft = _norm_failtime(ep.gt_failure_step)
    fsec = _secs(ft) if ft else None
    if fsec is None: return None
    cands = [(i, _secs(st.time)) for i, st in enumerate(ep.steps) if _secs(st.time) is not None]
    if not cands: return None
    le = [(i, s) for i, s in cands if s <= fsec]
    if le: return max(le, key=lambda x: x[1])[0]
    return min(cands, key=lambda x: abs(x[1] - fsec))[0]

def intended_effect(action):
    """Textbook (intended) effect of an action goal, regardless of whether it occurred.
    Returns (intended_true_phrase, intended_false_phrase) or None."""
    verb, objs = parse_goal(action)
    if not verb or not objs: return None
    x = objs[0]
    if verb in ("toggle on", "toggle off"):
        w = "turned on" if verb == "toggle on" else "turned off"
        return (f"the {x} is {w}", f"the {x} is {'turned off' if w=='turned on' else 'turned on'}")
    if verb in ("open", "close"):
        w = "open" if verb == "open" else "closed"
        return (f"the {x} is {w}", f"the {x} is {'closed' if w=='open' else 'open'}")
    if verb == "slice":   return (f"the {x} is sliced",  f"the {x} is not sliced")
    if verb == "crack":   return (f"the {x} is cracked", f"the {x} is not cracked")
    if verb == "pick up": return (f"the {x} is inside the robot gripper", "nothing is inside the robot gripper")
    if verb == "put in" and len(objs) > 1: return (f"the {x} is inside the {objs[1]}", f"the {objs[1]} is inside the {x}")
    if verb == "put on" and len(objs) > 1: return (f"the {x} is on top of the {objs[1]}", f"the {objs[1]} is on top of the {x}")
    if verb == "pour":    return (f"the {x} is empty", f"the {x} is filled with water and clean")
    return None

def actual_fact(step, obj):
    """The observed fact about obj at this step (state or first relation), or None."""
    if obj in step.states: return fmt_state(obj, step.states[obj])
    for r in step.relations:
        if r[0] == obj and r[0] != "nothing": return fmt_rel(r)
    return None

def state_at_other(ep, j, obj, value_excl):
    """A fact about obj observed at some step != j that does not hold at step j."""
    for k, st in enumerate(ep.steps):
        if k == j: continue
        f = actual_fact(st, obj)
        if f and f != value_excl and not entails(ep.steps[j], f):
            return f
    return None

def step_option(step):
    """A trace step rendered as a uniquely-identifiable answer option."""
    return f"{step.action.strip().lower()} (at {step.time})"

# Attribution: canonical, episode-instantiated failure-mode phrasings.
def _dropped_obj(reason):
    m = re.match(r"\s*dropped\s+(\w+)", reason or "", re.I)
    return m.group(1) if m else None

def failure_category(reason):
    r = (reason or "").lower()
    if not r: return "other"
    if "dropped" in r:                                                   return "dropped"
    if "occlud" in r or "occlusion" in r or "blocking" in r:            return "occlusion"
    if "instead of" in r or "mis-identified" in r or "misidentified" in r or "should use" in r or "wrong object" in r:
        return "wrong_object"
    if "wrong order" in r or " after " in r or " before " in r or "order" in r: return "order"
    if "precondition" in r:                                              return "precondition"
    if ("never" in r or "forgot" in r or "did not" in r or "didn't" in r
            or "failed to" in r or "missing" in r or "skip" in r):      return "missing"
    return "other"

def attribution_phrase(cat, obj_t):
    return {
        "dropped":      f"the robot dropped the {obj_t} during execution",
        "occlusion":    f"an object blocked the robot from reaching the {obj_t}",
        "wrong_object": "the robot acted on the wrong object",
        "missing":      "the robot skipped a required action",
        "order":        "the robot performed the actions in the wrong order",
        "precondition": "the robot attempted an action before its precondition was met",
    }.get(cat)

# Context regimes ------------------------------------------------------------
def ctx_full(ep):
    return " ".join(goal_line(s) + " " + obs_line(s) for s in ep.steps)

def ctx_window(ep, j, radius=2):
    lo, hi = max(0, j - radius), min(len(ep.steps), j + radius + 1)
    return " ".join(goal_line(s) + " " + obs_line(s) for s in ep.steps[lo:hi])

def ctx_plan(ep, steps=None):
    return f"Goal: {ep.success.rstrip('.')}. Plan: {plan_text(ep, steps)}."

In [ ]:
# question container + validity gate
@dataclass
class Question:
    qid:str; task:str; episode:str; rtype:str; regime:str
    context:str; stem:str; correct:str; distractors:list
    kinds:list = field(default_factory=list)
    options:dict = field(default_factory=dict); answer:str = ""
    meta:dict = field(default_factory=dict)

NONE_OPT = "an option not listed here"

def norm_txt(s): return re.sub(r"\s+", " ", str(s).strip().lower())

def _plan_items(ctx):
    return {s.strip() for s in re.findall(r"\d+\)\s*([^;.]+?)(?=;|\.|$)", ctx)}

_ORD = r"(?:first |second |third |fourth |fifth |sixth )?"
IMPOSSIBLE_RE = re.compile(
    r"^(?:slice the " + _ORD + r"(?:sink|countertop|counter top|stove burner|faucet|coffee machine|"
    r"fridge|microwave|toaster|dining table|tv stand|garbage can|house plant|wall)"
    r"|toggle (?:on|off) the " + _ORD + r"(?:bowl|plate|mug|cup|pan|kettle|knife|apple|tomato|potato|egg|pot|bread)"
    r"|pick up the " + _ORD + r"(?:sink|countertop|counter top|stove burner|faucet|coffee machine|"
    r"fridge|microwave|toaster|dining table|tv stand|garbage can|house plant|wall))")

def validate(q, ep):
    """Structural gate: well-formed, context-necessary, no gimme distractors. Returns bool.
    On success, trims q.distractors to the 3 retained options."""
    ds = list(dict.fromkeys([d for d in q.distractors if d and d != q.correct and d != NONE_OPT]))
    if len(ds) < 3: return False
    ds = ds[:3]
    if any(IMPOSSIBLE_RE.match(norm_txt(d)) for d in ds): return False     # no impossible gimmes
    if norm_txt(q.correct) in norm_txt(q.stem): return False                # answer not lifted from stem
    if q.rtype == "T1":
        st = ep.steps[q.meta["step"]]
        if not entails(st, q.correct): return False                        # correct truly holds
        if any(entails(st, d) for d in ds): return False                   # every distractor false
        if q.meta.get("trap") not in ds: return False                      # the prior answer is an option
    if q.rtype == "T4":
        items = {norm_txt(x) for x in _plan_items(q.context)}
        if norm_txt(q.correct) in items: return False                      # missing step must be absent
        if not any(norm_txt(d) in items for d in ds): return False         # a present-step trap exists
    q.distractors = ds
    return True

def finalize(q, ep, seed):
    """Run the gate, then shuffle correct + 3 distractors + 'none' into A-E."""
    if not validate(q, ep): return None
    texts = [q.correct] + q.distractors[:3] + [NONE_OPT]
    if len(set(texts)) != 5: return None
    rng = random.Random(f"{seed}|{q.qid}")
    rng.shuffle(texts)
    letters = ["A","B","C","D","E"]
    q.options = {letters[i]: texts[i] for i in range(5)}
    q.answer = letters[texts.index(q.correct)]
    return q

In [ ]:
# failure-centric generators (T1-T4)

def _mk(ep, rtype, regime, idx, ctx, stem, correct, distractors, kinds, meta):
    return Question(f"{ep.name}|{rtype}|{regime}|{idx}", ep.task, ep.name, rtype, regime,
                    ctx, stem, correct, distractors, kinds, meta=meta)

def gen_T1(ep, regimes, cap, seed):
    """Outcome verification. Ask what was ACTUALLY observed after an action whose intended
    effect did not occur (a divergence). The textbook effect is the trap distractor, so a
    context-free guesser is wrong and the context is load-bearing by construction."""
    out, div = [], []
    for j, st in enumerate(ep.steps):
        ie = intended_effect(st.action)
        if not ie: continue
        _, objs = parse_goal(st.action); x = objs[0]
        act = actual_fact(st, x)
        if not act or act == ie[0] or entails(st, ie[0]): continue   # intended happened -> trivial
        div.append((j, x, act, ie[0], ie[1]))
    for reg in regimes:
        n = 0
        for (j, x, act, itrue, ifalse) in div:
            if n >= cap: break
            st = ep.steps[j]
            _, objs = parse_goal(st.action)
            alt = alt_object(ep, parse_goal(st.action)[0], x, objs[1] if len(objs) > 1 else None, seed + j)
            d_obj = act.replace(f"the {x} ", f"the {alt} ", 1) if alt and f"the {x} " in act else None
            d_time = state_at_other(ep, j, x, act)
            cand = [c for c in [itrue, d_obj, d_time, ifalse] if c and c != act and not entails(st, c)]
            ctx = ctx_full(ep) if reg == "C1" else ctx_window(ep, j)
            stem = (f"The robot tried to {st.action.strip().lower()} at {st.time}. "
                    "What was actually observed afterward?")
            q = _mk(ep, "T1", reg, j, ctx, stem, act, cand,
                    ["intended_trap","wrong_object","wrong_timestep"], {"step": j, "trap": itrue})
            fq = finalize(q, ep, seed)
            if fq: out.append(fq); n += 1
    return out

def gen_T2(ep, regimes, cap, seed):
    """Failure localization: which executed step did the run first go wrong at. Anchored on
    gt_failure_step; distractors are other real steps, so the trace must be read."""
    out = []
    j = failure_step_index(ep)
    if j is None: return out
    correct = step_option(ep.steps[j])
    distract = []
    for k in sorted(range(len(ep.steps)), key=lambda k: abs(k - j)):
        if k == j: continue
        opt = step_option(ep.steps[k])
        if opt != correct and opt not in distract: distract.append(opt)
        if len(distract) >= 6: break
    for reg in regimes:
        ctx = ctx_full(ep)
        stem = ("The robot did not achieve its goal: " + ep.success.rstrip(".") +
                ". At which step did execution first go wrong?")
        q = _mk(ep, "T2", reg, j, ctx, stem, correct, distract[:3],
                ["other_step","other_step","other_step"], {"step": j})
        fq = finalize(q, ep, seed)
        if fq: out.append(fq)
    return out

def gen_T3(ep, regimes, cap, seed):
    """Failure attribution: why the task failed. Correct is a canonical phrasing of the
    gt_failure_reason category; distractors are other failure modes for this episode."""
    out = []
    cat = failure_category(ep.gt_failure_reason)
    if cat == "other": return out
    j = failure_step_index(ep)
    dobj = _dropped_obj(ep.gt_failure_reason)
    obj_t = obj_text(dobj) if dobj else None
    if obj_t is None and j is not None:
        _, os_ = parse_goal(ep.steps[j].action); obj_t = os_[0] if os_ else None
    obj_t = obj_t or "object"
    correct = attribution_phrase(cat, obj_t)
    if not correct: return out
    others = [c for c in ["dropped","occlusion","wrong_object","missing","order","precondition"] if c != cat]
    so = sorted(scene_objs(ep))
    rng = random.Random(f"{seed}|{ep.name}|T3"); rng.shuffle(others); rng.shuffle(so)
    alt_o = next((o for o in so if base_obj(o) in PICKABLE_POOL and o != obj_t), obj_t)
    distract = []
    for c in others:
        ph = attribution_phrase(c, alt_o if c in ("dropped","occlusion") else obj_t)
        if ph and ph != correct and ph not in distract: distract.append(ph)
    for reg in regimes:
        ctx = ctx_full(ep) if reg == "C1" else ctx_window(ep, j if j is not None else len(ep.steps)//2)
        stem = f"The robot's goal was: {ep.success.rstrip('.')}. It did not succeed. What best explains the failure?"
        q = _mk(ep, "T3", reg, 0, ctx, stem, correct, distract[:3],
                ["failure_mode","failure_mode","failure_mode"], {"step": j})
        fq = finalize(q, ep, seed)
        if fq: out.append(fq)
    return out

def gen_T4(ep, regimes, cap, seed):
    """Missing-step. Prefer GT specified_missing_steps (verified-required); else remove a
    uniquely-occurring required interact step. Distractors are a present step, a wrong-object
    variant, and a valid-but-not-required action (no impossible options)."""
    out = []
    flat = []
    for grp in (ep.specified_missing_steps or []):
        flat += grp if isinstance(grp, list) else [grp]
    flat = [i for i in flat if isinstance(i, int) and 0 <= i < len(ep.plan) and is_interact(ep.plan[i])]
    if flat:
        cand_idx, kind0 = list(dict.fromkeys(flat)), "gt_missing"
    else:
        inter = [(k, a) for k, a in enumerate(ep.plan) if is_interact(a)]
        texts = [action_text(a) for _, a in inter]
        cand_idx, kind0 = [k for (k, a), t in zip(inter, texts) if texts.count(t) == 1], "unique_missing"
    extra_pool = [f"toggle off the {o}" for o in scene_objs(ep) if base_obj(o) in TOGGLABLE_POOL]
    for reg in regimes:
        n = 0
        for i in cand_idx:
            if n >= cap: break
            reduced = [a for k, a in enumerate(ep.plan) if k != i]
            correct = action_text(ep.plan[i])
            verb, args = parse_action(ep.plan[i]); vphrase = _VERB.get(verb, ("",))[0]
            alt = alt_object(ep, vphrase, obj_text(args[0]), obj_text(args[1]) if len(args) > 1 else None, seed + i)
            d_obj = (f"{vphrase} the {alt}" if verb not in ("put_in","put_on","pour")
                     else correct.replace(f"the {obj_text(args[0])}", f"the {alt}", 1)) if alt else None
            present = [action_text(a) for a in reduced if is_interact(a)]
            d_present = next((p for p in present if p != correct and p != d_obj), None)
            d_extra = next((x for x in extra_pool if x not in (correct, d_obj, d_present)), None)
            cand = [d_present, d_obj, d_extra]
            if reg == "C2":
                eff = next((s for s in ep.steps if goal_phrase(s.action) == correct), None)
                ctx = ctx_plan(ep, reduced) + (f" Around the gap it was observed that {render_obs(eff)}." if eff else "")
            else:
                ctx = ctx_plan(ep, reduced)
            stem = (f"The plan below was meant to achieve: {ep.success.rstrip('.')}. "
                    "Exactly one required step has been removed. Which step is missing?")
            q = _mk(ep, "T4", reg, i, ctx, stem, correct, cand,
                    [kind0,"wrong_object","not_required"], {"plan_idx": i})
            fq = finalize(q, ep, seed)
            if fq: out.append(fq); n += 1
    return out

GRID = {"T1": ["C1","C2"], "T2": ["C1"], "T3": ["C1","C2"], "T4": ["C2","C3"], }
GENS = {"T1": gen_T1, "T2": gen_T2, "T3": gen_T3, "T4": gen_T4, }
RTYPE_NAME  = {"T1":"outcome verification","T2":"failure localization","T3":"failure attribution",
               "T4":"missing step",}
REGIME_NAME = {"C1":"full trace","C2":"local window","C3":"plan only"}
RTYPES  = list(GRID)
REGIMES = ["C1","C2","C3"]

def generate_all(eps, cap=3, seed=16):
    qs = []
    for ep in eps:
        for t, regs in GRID.items():
            qs.extend(GENS[t](ep, regs, cap, seed))
    return qs

questions = generate_all(episodes, cap=CAP_PER_CELL, seed=SEED)
print(f"generated {len(questions)} questions across {sum(len(v) for v in GRID.values())} grid cells")
print("by cell:", dict(sorted(Counter(f"{q.rtype}x{q.regime}" for q in questions).items())))

## 3b. Validity gate and audit

Quality is enforced at generation time by `validate()`, not only checked after the fact. An
item is kept only if it has five distinct options including the answer, no impossible
distractor, an answer that is not lifted from the stem, and a single defensible answer (for
T1 the answer is entailed at its step and every distractor is false; for T4 the missing step
is absent from the shown plan and a present-step trap exists). An empirical context-ablation
gate (Section 5) then drops any item the five-LLM panel can answer without its context, so
the retained set is context-necessary. The cell below re-audits the final set.

In [ ]:
ep_by_name = {ep.name: ep for ep in episodes}

problems = Counter()
for q in questions:
    opts = [q.options[L] for L in "ABCDE"]
    if len(set(opts)) != 5:              problems["duplicate_options"] += 1
    if q.options[q.answer] != q.correct: problems["answer_mismatch"] += 1
    if NONE_OPT not in opts:             problems["missing_none_option"] += 1
    if any(IMPOSSIBLE_RE.match(norm_txt(o)) for o in opts): problems["impossible_distractor"] += 1
    if norm_txt(q.correct) in norm_txt(q.stem):            problems["answer_in_stem"] += 1
    ep = ep_by_name[q.episode]
    distract = [q.options[L] for L in "ABCDE" if q.options[L] not in (q.correct, NONE_OPT)]
    if q.rtype == "T1":
        st = ep.steps[q.meta["step"]]
        if not entails(st, q.correct):            problems["T1_answer_not_true"] += 1
        if any(entails(st, d) for d in distract): problems["T1_distractor_true"] += 1
    if q.rtype == "T4":
        items = {norm_txt(x) for x in _plan_items(q.context)}
        if norm_txt(q.correct) in items:          problems["T4_missing_shown"] += 1

print("audited", len(questions), "questions")
print("validity problems:", dict(problems) if problems else "NONE, all checks pass")
assert not problems, "question set failed validity audit"

## 4. Model interface

The option score is the model's next-token probability for the option letter, normalised over
the five letters (KnowNo). From that option distribution we record two quantities. **`max_prob`**
is the probability mass on the chosen (arg-max) option - the raw KnowNo confidence, kept for
reference. **`norm_entropy`** is the Shannon entropy of the option distribution divided by
`log(#options)`, so it lands in `[0, 1]` (0 = all mass on one option, 1 = uniform). To match
`sim_uncertainty_calibration_standard.ipynb` - whose MMLU-Pro items have a varying number of
options - the **calibration confidence used in every metric and figure below is the derived
certainty `confidence = 1 - norm_entropy`**. Here every item has 5 options, so the normaliser
`log(5)` is constant, but using the same definition keeps the two notebooks directly comparable.
Responses are cached on disk keyed by (model, prompt); the raw logprob fields are cached and the
entropy is derived on read, so the definition can change without re-querying. Sampling is greedy
(`temperature=0`, one token), so calls are deterministic.

In [ ]:
ANSWER_SYS = ("You answer multiple-choice questions about a robot's task execution. "
              "Read the context and output ONLY the single letter (A-E) of the best option. "
              "No words, no punctuation, no explanation.")

def build_user(q):
    opts = "\n".join(f"{L} = {q.options[L]}" for L in "ABCDE")
    return f"Context: {q.context}\n\nQuestion: {q.stem}\n{opts}\nAnswer:"

def build_user_noctx(q):
    # Same prompt with the context removed: used by the context-ablation gate.
    opts = "\n".join(f"{L} = {q.options[L]}" for L in "ABCDE")
    return f"Question: {q.stem}\n{opts}\nAnswer:"

def build_prompter(model):
    """Return a prompter for a model. qwen (local) and gpt-5.4 (portkey) score from
    logprobs; claude (anthropic) has no logprobs, so AnthropicLLMPrompter estimates the
    option distribution by sampling. Raises if a required API key is missing."""
    cfg = MODEL_BACKENDS.get(model, {"backend": "local"})
    backend = cfg["backend"]
    if backend == "local":
        return LocalLLMPrompter(model_name=model, base_url=OLLAMA_URL, think=False)
    if backend == "portkey":
        key = os.environ.get("PORTKEY_API_KEY")  # set in your shell or .env (see .env.example)
        if not key: raise RuntimeError("PORTKEY_API_KEY not set")
        vk = os.environ.get(cfg.get("virtual_key_env", "")) or cfg.get("virtual_key")
        return PortkeyLLMPrompter(portkey_api_key=key, model=model, virtual_key=vk,
                                  reasoning_effort=cfg.get("reasoning_effort"))
    raise ValueError(f"unknown backend {backend!r}")

_CACHE_PATH = OUT_DIR / "llm_cache.json"
_LLM_CACHE = json.loads(_CACHE_PATH.read_text()) if _CACHE_PATH.exists() else {}

def _cache_key(model, user):
    return hashlib.sha256(f"{model}\x00{ANSWER_SYS}\x00{user}".encode()).hexdigest()

def _flush_cache():
    _CACHE_PATH.write_text(json.dumps(_LLM_CACHE))

def normalized_entropy(option_probs):
    """Shannon entropy of the option distribution, normalised to [0, 1] by log(#options).
    Matches sim_uncertainty_calibration_standard.ipynb so the two notebooks are comparable:
    0 = all probability mass on one option, 1 = uniform. Returns None if no distribution."""
    if not option_probs:
        return None
    p = np.asarray(list(option_probs.values()), dtype=float)
    n = p.size
    if n <= 1:
        return 0.0
    p = p[p > 0]
    if p.size == 0:
        return None
    raw = -float(np.sum(p * np.log(p)))     # nats
    return raw / np.log(n)

def ask(prompter, model, q, user=None):
    """Score one question. Returns predicted_label, option_probs, max_prob (KnowNo confidence),
    norm_entropy, confidence (= 1 - norm_entropy, the calibration confidence), score_status,
    correct. Backend-agnostic: reads score_metadata produced from logprobs OR from sampling.
    Pass user=build_user_noctx(q) to score the context-ablated prompt."""
    user = build_user(q) if user is None else user
    key = _cache_key(model, user)
    if key not in _LLM_CACHE:
        cs = {L: q.options[L] for L in "ABCDE"}
        try:
            _txt, meta = prompter.query(prompt={"system": ANSWER_SYS, "user": user},
                                        sampling_params={"max_tokens": 1, "temperature": 0.0, "top_p": 1},
                                        choice_spec=cs)
            _LLM_CACHE[key] = {"predicted_label": meta.get("predicted_label"),
                               "confidence": meta.get("confidence"),
                               "option_probs": meta.get("option_probs"),
                               "score_status": meta.get("score_status")}
        except Exception as e:
            _LLM_CACHE[key] = {"predicted_label": None, "confidence": None,
                               "option_probs": None, "score_status": f"error:{type(e).__name__}"}
    m = _LLM_CACHE[key]
    ne = normalized_entropy(m.get("option_probs")) if m.get("option_probs") else None
    return {"predicted_label": m["predicted_label"], "max_prob": m["confidence"],
            "option_probs": m["option_probs"], "score_status": m["score_status"],
            "norm_entropy": ne, "confidence": (None if ne is None else 1.0 - ne),
            "correct": int(m["predicted_label"] == q.answer) if m["predicted_label"] else 0}

def run_model_over_questions(model, qs, label):
    """Iterate one model over all questions, reusing the on-disk cache."""
    prompter = build_prompter(model)
    t0 = time.time(); res = {}
    for k, q in enumerate(qs):
        res[q.qid] = ask(prompter, model, q)
        if (k + 1) % 40 == 0:
            print(f"  [{label}] {k+1}/{len(qs)}", flush=True)
    _flush_cache()
    print(f"  [{label}] done {len(qs)} in {time.time()-t0:.0f}s")
    return res

## 5. Difficulty labelling

A panel of five LLMs (none of them the model under test) answers each question once with
no reasoning steps. A question's **panel solvability** is the number of correct panel
answers (0-5), used as a difficulty proxy for contemporary LLMs.

In [ ]:
ABLATION_GATE = 1
NCTX_TRIVIAL  = 4    # drop items >= this many of the 5 panel models get right with no context

panel_correct = defaultdict(int)
for model in PANEL_MODELS:
    print(f"panel (with context): {model}")
    res = run_model_over_questions(model, questions, model)
    for qid, r in res.items():
        panel_correct[qid] += r["correct"]

if ABLATION_GATE:
    noctx_correct = defaultdict(int)
    for model in PANEL_MODELS:
        print(f"panel (no context): {model}")
        prompter = build_prompter(model)
        for q in questions:
            noctx_correct[q.qid] += ask(prompter, model, q, user=build_user_noctx(q))["correct"]
        _flush_cache()
    before = len(questions)
    before_questions = questions.copy()
    questions = [q for q in questions if noctx_correct[q.qid] < NCTX_TRIVIAL]
    print(f"context-ablation gate: dropped {before - len(questions)}/{before} items answerable "
          f"without context (>= {NCTX_TRIVIAL}/5 panel correct blind); {len(questions)} remain")
    print(f"dropped by cell:", dict(sorted(Counter(f"{q.rtype}x{q.regime}" for q in before_questions if noctx_correct[q.qid] >= NCTX_TRIVIAL).items())))

solvability = {q.qid: panel_correct[q.qid] for q in questions}
print("solvability distribution (0..5):", dict(sorted(Counter(solvability.values()).items())))

## 6. Model under test

The model under test answers every retained question. For each answer we record the option
probabilities, the chosen option's confidence, and whether the chosen option is correct.

In [ ]:
print("models under test:", MUT_MODELS)
all_rows = []
for model in MUT_MODELS:
    try:
        res = run_model_over_questions(model, questions, model)
    except Exception as e:
        print(f"  [skip] {model}: {type(e).__name__}: {e}")
        continue
    for q in questions:
        r = res[q.qid]; solv = solvability[q.qid]
        all_rows.append({"model": model, "qid": q.qid, "task": q.task, "episode": q.episode,
                         "rtype": q.rtype, "regime": q.regime, "cell": f"{q.rtype}x{q.regime}",
                         "answer": q.answer, "predicted": r["predicted_label"],
                         "max_prob": r["max_prob"], "norm_entropy": r["norm_entropy"],
                         "confidence": r["confidence"], "correct": r["correct"],
                         "score_status": r["score_status"], "solvability": solv,
                         "difficulty": difficulty_bucket(solv)})

df_all = pd.DataFrame(all_rows)
df_all.to_csv(OUT_DIR / "results_all_models.csv", index=False)
print("\nper-model coverage:")
print(df_all.groupby("model").agg(n=("qid", "size"),
      with_conf=("confidence", lambda s: s.notna().sum()),
      accuracy=("correct", "mean"),
      mean_entropy=("norm_entropy", "mean")).round(3).to_string())

df_cal = df_all[df_all["confidence"].notna()].copy()
df_cal.head()

In [ ]:
# === Attach difficulty-balanced weights (composition-neutral target distribution) =
# The generator over-produces easy/saturated items, so the as-generated mix is a
# property of the engine, not of the deployment task. We weight every item so each
# panel-solvability level (0..5) contributes equally -- a principled "target
# distribution" -- mirroring the inverse-probability weights in the standard
# (MMLU-Pro) notebook. Within a difficulty level the weight is constant, so only
# pooled/marginal metrics move. (See also the Composition-sensitivity section, which
# uses the same balancing idea as a robustness check.)
_cnt = df_all.groupby("solvability")["qid"].transform("count")
_G = df_all["solvability"].nunique()
df_all["weight"] = len(df_all) / (_G * _cnt)
df_cal = df_all[df_all["confidence"].notna()].copy()
_w = df_cal["weight"].values
_e = (_w.sum() ** 2) / np.sum(_w ** 2)
print(f"difficulty-balanced weights attached; solvability levels="
      f"{sorted(int(s) for s in df_all.solvability.unique())}; Kish ESS={_e:.0f}/{len(df_cal)} "
      f"(design effect={len(df_cal)/_e:.2f}).")


In [ ]:
# Retry ONLY the gpt-5.4 questions that failed (cached errors / missing / no confidence)

MODEL = "gpt-5.4"
RETRIES = 5            # attempts per question (transient APIConnectionError, etc.)

def _needs_redo(entry):
    return (entry is None
            or str(entry.get("score_status", "")).startswith("error")
            or entry.get("confidence") is None)

todo = [q for q in questions if _needs_redo(_LLM_CACHE.get(_cache_key(MODEL, build_user(q))))]
print(f"{len(todo)} gpt-5.4 questions need redo (of {len(questions)})")

prompter = build_prompter(MODEL)     # make sure MODEL_BACKENDS['gpt-5.4'] reasoning_effort='none' is live
sp = {"max_tokens": 1, "temperature": 0.0, "top_p": 1}
fixed, still_failing = 0, []
for i, q in enumerate(todo):
    key = _cache_key(MODEL, build_user(q))
    cs = {L: q.options[L] for L in "ABCDE"}
    err = None
    for attempt in range(RETRIES):
        try:
            _txt, meta = prompter.query(prompt={"system": ANSWER_SYS, "user": build_user(q)},
                                        sampling_params=sp, choice_spec=cs)
            _LLM_CACHE[key] = {"predicted_label": meta.get("predicted_label"),
                               "confidence":      meta.get("confidence"),
                               "option_probs":    meta.get("option_probs"),
                               "score_status":    meta.get("score_status")}
            err = None; fixed += 1; break
        except Exception as e:
            err = e
            time.sleep(2 * (attempt + 1))      # 2s, 4s, 6s, 8s backoff
    if err is not None:
        still_failing.append((q.qid, type(err).__name__))
    if (i + 1) % 25 == 0:
        _flush_cache(); print(f"  {i+1}/{len(todo)} done")

_flush_cache()
print(f"redo complete: {fixed}/{len(todo)} now answered; {len(still_failing)} still failing")
if still_failing:
    print("still failing:", still_failing[:10])

## 7. Calibration metrics and two-level bootstrap

Expected Calibration Error (ECE) bins predictions by confidence and averages the
|accuracy - confidence| gap weighted by bin count. The reliability curve plots accuracy
against mean confidence per bin. Because the data are clustered by task and by episode,
confidence intervals come from a **two-level bootstrap**: resample tasks with replacement,
then resample episodes within each chosen task, and recompute the statistic.

In [ ]:
def ece(conf, correct, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    edges = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        total += (m.sum() / len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return total

def reliability_curve(conf, correct, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    edges = np.linspace(0, 1, n_bins + 1)
    pts = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        pts.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    return pts  # list of (mean_conf, accuracy, count)

def m_accuracy(d):   return d["correct"].mean() if len(d) else np.nan
def m_meanconf(d):   return d["confidence"].mean() if len(d) else np.nan
def m_ece(d):        return ece(d["confidence"].values, d["correct"].values) if len(d) else np.nan

def two_level_bootstrap(d, metric_fn, B=N_BOOT, seed=SEED):
    """Resample tasks with replacement, then episodes within each task; recompute metric_fn."""
    if len(d) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    tasks = d["task"].unique()
    eps_by_task = {t: d[d.task == t]["episode"].unique() for t in tasks}
    rows_by_ep = {(t, e): d[(d.task == t) & (d.episode == e)] for t in tasks for e in eps_by_task[t]}
    vals = []
    for _ in range(B):
        parts = []
        for t in rng.choice(tasks, size=len(tasks), replace=True):
            eps = eps_by_task[t]
            for idx in rng.integers(0, len(eps), size=len(eps)):
                parts.append(rows_by_ep[(t, eps[idx])])
        bs = pd.concat(parts, ignore_index=True)
        v = metric_fn(bs)
        if v is not None and not (isinstance(v, float) and math.isnan(v)):
            vals.append(v)
    if not vals: return (np.nan, np.nan, np.nan)
    return tuple(np.percentile(vals, [2.5, 50, 97.5]))

def summarise(d):
    out = {"n": len(d), "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d), "ece": m_ece(d),
           "mean_entropy": d["norm_entropy"].mean() if len(d) else np.nan,
           "gap": (m_meanconf(d) - m_accuracy(d))}
    for name, fn in [("accuracy", m_accuracy), ("ece", m_ece)]:
        lo, mid, hi = two_level_bootstrap(d, fn)
        out[f"{name}_lo"], out[f"{name}_hi"] = lo, hi
    return out

print(f"overall: accuracy={m_accuracy(df_cal):.3f}  mean_conf={m_meanconf(df_cal):.3f}  ECE={m_ece(df_cal):.3f}")

In [ ]:
# === Proper scoring rules + weighting (added) ============================
# ECE is a *binned, non-proper* summary of top-label calibration: a model can move
# items between bins without changing it. We add two strictly *proper* scoring rules
# on the SAME (confidence, correct) signal ECE uses, so the model ordering cannot be
# an artefact of the binning:
#   Brier = mean_w (confidence - correct)^2            (bounded in [0,1])
#   NLL   = -mean_w [y*log(p) + (1-y)*log(1-p)]         (log loss; punishes confident errors)
# Both are linear in the samples, so weighting is exact and low-variance (unlike ECE,
# whose bins interact with the weights). All functions take an optional `weight=`;
# weight=None reproduces the original unweighted behaviour, so existing figures are
# unaffected. These defs intentionally OVERRIDE the ones above with weighted-capable
# supersets used by every cell below.
EPS = 1e-12

def wmean(x, w=None):
    x = np.asarray(x, float)
    if w is None: return float(x.mean()) if x.size else np.nan
    w = np.asarray(w, float)
    return np.nan if w.sum() == 0 else float(np.average(x, weights=w))

def ess(w):
    """Kish effective sample size; ESS/n = 1/design-effect quantifies variance inflation."""
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / np.sum(w ** 2)) if w.sum() > 0 else 0.0

def ece(conf, correct, n_bins=ECE_BINS, weight=None):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    w = np.ones_like(conf) if weight is None else np.asarray(weight, float)
    W = w.sum()
    if W == 0: return np.nan
    edges = np.linspace(0, 1, n_bins + 1); total = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        wb = w[m].sum()
        if wb == 0: continue
        total += (wb / W) * abs(wmean(correct[m], w[m]) - wmean(conf[m], w[m]))
    return total

def brier(conf, correct, weight=None):
    conf = np.asarray(conf, float); y = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    return wmean((conf - y) ** 2, None if weight is None else np.asarray(weight, float))

def nll(conf, correct, weight=None):
    conf = np.clip(np.asarray(conf, float), EPS, 1 - EPS); y = np.asarray(correct, float)
    if len(conf) == 0: return np.nan
    ll = y * np.log(conf) + (1 - y) * np.log(1 - conf)
    return -wmean(ll, None if weight is None else np.asarray(weight, float))

# dataframe wrappers (the slice must carry a 'weight' column); w=True -> weighted
def m_accuracy(d, w=False): return wmean(d["correct"].values, d["weight"].values if w else None)
def m_meanconf(d, w=False): return wmean(d["confidence"].values, d["weight"].values if w else None)
def m_ece(d, w=False):      return ece(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)
def m_brier(d, w=False):    return brier(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)
def m_nll(d, w=False):      return nll(d["confidence"].values, d["correct"].values, weight=d["weight"].values if w else None)

_BOOT = two_level_bootstrap   # bootstrap function defined in the metrics cell above

def summarise(d):
    out = {"n": len(d), "ess": round(ess(d["weight"].values), 1),
           "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d),
           "ece": m_ece(d), "brier": m_brier(d), "nll": m_nll(d),
           "ece_w": m_ece(d, True), "brier_w": m_brier(d, True), "nll_w": m_nll(d, True),
           "mean_entropy": d["norm_entropy"].mean() if (len(d) and "norm_entropy" in d) else np.nan,
           "gap": (m_meanconf(d) - m_accuracy(d))}
    for name, fn in [("accuracy", m_accuracy), ("ece", m_ece)]:
        lo, mid, hi = _BOOT(d, fn)
        out[f"{name}_lo"], out[f"{name}_hi"] = lo, hi
    return out

print("proper-scoring metrics ready: ece (now weight-capable), brier, nll; "
      "summarise() now reports brier/nll and weighted variants.")


## 8. Results per grid cell and per difficulty bucket

In [ ]:
# Per grid cell (reasoning type x context regime).
cell_rows = []
for t in GRID:
    for reg in GRID[t]:
        d = df_cal[(df_cal.rtype == t) & (df_cal.regime == reg)]
        if len(d) == 0: continue
        s = summarise(d)
        cell_rows.append({"rtype": t, "regime": reg, **s})
cell_tbl = pd.DataFrame(cell_rows)
cell_tbl_disp = cell_tbl.assign(
    accuracy=lambda x: x.accuracy.round(3), mean_conf=lambda x: x.mean_conf.round(3),
    ece=lambda x: x.ece.round(3), gap=lambda x: x.gap.round(3),
    mean_entropy=lambda x: x.mean_entropy.round(3),
    acc_CI=lambda x: list(zip(x.accuracy_lo.round(2), x.accuracy_hi.round(2))),
    ece_CI=lambda x: list(zip(x.ece_lo.round(2), x.ece_hi.round(2))),
)[["rtype","regime","n","accuracy","acc_CI","mean_conf","ece","ece_CI","mean_entropy","gap"]]
print("Per grid cell (gap = mean confidence - accuracy; positive = overconfident):")
cell_tbl_disp

In [ ]:
# Per reasoning type and per regime (marginals), and per difficulty bucket.
def marginal_table(group_col, order):
    rws = []
    for g in order:
        d = df_cal[df_cal[group_col] == g]
        if len(d) == 0: continue
        rws.append({group_col: g, **summarise(d)})
    return pd.DataFrame(rws)

by_type   = marginal_table("rtype", list(GRID))
by_regime = marginal_table("regime", REGIMES)
by_diff   = marginal_table("difficulty", ["hard","medium","easy"])

def show(tbl, title):
    print(title)
    cols = [tbl.columns[0], "n", "accuracy", "mean_conf", "ece", "mean_entropy", "gap"]
    disp = tbl[cols].copy()
    for c in ["accuracy","mean_conf","ece","mean_entropy","gap"]: disp[c] = disp[c].round(3)
    print(disp.to_string(index=False)); print()

show(by_type,   "By reasoning type:")
show(by_regime, "By context regime:")
show(by_diff,   "By difficulty bucket (panel solvability):")

## 9. Reliability diagrams and calibration figures

In [ ]:
# Multi-model setup for all figures below
cal_all = df_all[df_all["confidence"].notna() & (df_all["score_status"] == "available")].copy()
present = [m for m in MUT_MODELS if m in cal_all["model"].unique()]
MCOLOR  = {"qwen3.5:9b": "#1f77b4", "qwen3.5:27b": "#ff7f0e", "gpt-5.4": "#2ca02c"}
MCOLOR  = {m: MCOLOR.get(m, c) for m, c in zip(present, ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd"])} | MCOLOR

# FIG: reliability grid (model rows x facet cols). Marker AREA is scaled by the
#    number of items in each confidence bin, so sparse bins (little evidence) look
#    small and you do not over-read them. ECE already weights bins by count, so the
#    plot and the summary metric agree; the sizing is an explicit evidence cue, not
#    just an interpretation. ──────────────────────────────────────────────────
def reliability_grid(facet, levels, level_name, fname, title):
    nrow, ncol = len(present), len(levels)
    smax = 1
    for m in present:
        for lv in levels:
            d = cal_all[(cal_all.model == m) & (cal_all[facet] == lv)]
            for _, _, c in reliability_curve(d.confidence.values, d.correct.values): smax = max(smax, c)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 3.1*nrow), squeeze=False)
    for r, m in enumerate(present):
        for c, lv in enumerate(levels):
            ax = axes[r][c]; d = cal_all[(cal_all.model == m) & (cal_all[facet] == lv)]
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            pts = reliability_curve(d.confidence.values, d.correct.values)
            if pts:
                xs, ys, cs = zip(*pts)
                ax.plot(xs, ys, "-", color=MCOLOR[m], lw=1, alpha=.5, zorder=2)
                ax.scatter(xs, ys, s=[25 + 275*(cc/smax) for cc in cs], color=MCOLOR[m],
                           edgecolor="white", linewidth=.5, zorder=3)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
            ax.set_title(f"acc={d.correct.mean():.2f} conf={d.confidence.mean():.2f}\nECE={ece(d.confidence.values, d.correct.values):.2f} n={len(d)}", fontsize=8)
            if c == 0: ax.set_ylabel(f"{m}\naccuracy", fontsize=9)
            if r == nrow-1: ax.set_xlabel(f"mean confidence\n{lv} {level_name.get(lv,'')}", fontsize=9)
            ax.tick_params(labelsize=7)
    fig.suptitle(title + "  (marker area ∝ #items in bin)", y=1.002, fontsize=12)
    fig.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=130, bbox_inches="tight"); plt.show()

reliability_grid("regime", REGIMES, REGIME_NAME, "reliability_grid_regime.png",
                 "Reliability by model x context regime")
reliability_grid("rtype", RTYPES, RTYPE_NAME, "reliability_grid_type.png",
                 "Reliability by model x reasoning type")

In [ ]:
# FIG: calibration-gap heatmap, one panel per model (shared diverging scale)
rtypes, regimes = RTYPES, REGIMES
grids = {}
for m in present:
    g = np.full((len(rtypes), len(regimes)), np.nan)
    for ri, t in enumerate(rtypes):
        for ci, reg in enumerate(regimes):
            d = cal_all[(cal_all.model == m) & (cal_all.rtype == t) & (cal_all.regime == reg)]
            if len(d): g[ri, ci] = d.confidence.mean() - d.correct.mean()
    grids[m] = g
vmax = np.nanmax([np.nanmax(np.abs(g)) for g in grids.values()])
fig, axes = plt.subplots(1, len(present), figsize=(4.6*len(present), 4.4), squeeze=False)
for c, m in enumerate(present):
    ax = axes[0][c]; g = grids[m]; im = ax.imshow(g, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(regimes))); ax.set_xticklabels([f"{r}\n{REGIME_NAME[r]}" for r in regimes], fontsize=8)
    # Only add y-tick labels to the leftmost column for readability
    if c == 0:
        ax.set_yticks(range(len(rtypes))); ax.set_yticklabels([f"{t}\n{RTYPE_NAME[t]}" for t in rtypes], fontsize=8)
    else:
        ax.set_yticks([])  # Hide y-tick labels for other columns
    for i in range(len(rtypes)):
        for j in range(len(regimes)):
            if np.isfinite(g[i, j]): ax.text(j, i, f"{g[i,j]:+.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(m, fontsize=11)
fig.colorbar(im, ax=axes[0].tolist(), fraction=0.025, pad=0.02, label="confidence - accuracy")
fig.suptitle("Calibration gap (positive = overconfident); blank = omitted cell", y=1.02, fontsize=12)
fig.savefig(OUT_DIR / "calibration_gap_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIG: accuracy / mean confidence / ECE vs difficulty, all models
solv = sorted(cal_all.solvability.unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (metric, fn, clip01) in zip(axes, [("accuracy", lambda d: d.correct.mean(), True),
                                           ("mean confidence", lambda d: d.confidence.mean(), True),
                                           ("ECE", lambda d: ece(d.confidence.values, d.correct.values), False)]):
    for m in present:
        ys = [fn(cal_all[(cal_all.model == m) & (cal_all.solvability == s)]) for s in solv]
        ax.plot(solv, ys, "-o", color=MCOLOR[m], label=m)
    ax.set_xlabel("panel solvability (0 = hardest, 5 = easiest)"); ax.set_title(metric); ax.grid(alpha=.3)
    ax.set_ylim(0, 1.02) if clip01 else ax.set_ylim(0, None)
ns = [int((cal_all[cal_all.model == present[0]].solvability == s).sum()) for s in solv]
axes[0].legend(fontsize=8)
axes[0].set_xticks(solv); axes[0].set_xticklabels([f"{s}\n(n={n})" for s, n in zip(solv, ns)], fontsize=8)
fig.suptitle("Accuracy / confidence / ECE vs difficulty, by model", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(OUT_DIR / "metrics_vs_difficulty_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# EXTRA FIGS: (a) ECE heatmap model x 13 cells, (b) calibration map scatter,
#    (c) confidence histograms. These pinpoint WHERE each model is miscalibrated. ──
cells = [f"{t}x{r}" for t in GRID for r in GRID[t]]

# (a) ECE heatmap (model x cell)
H = np.full((len(present), len(cells)), np.nan)
for ri, m in enumerate(present):
    for ci, cl in enumerate(cells):
        d = cal_all[(cal_all.model == m) & (cal_all.cell == cl)]
        if len(d): H[ri, ci] = ece(d.confidence.values, d.correct.values)
fig, ax = plt.subplots(figsize=(13, 3.2))
im = ax.imshow(H, cmap="OrRd", vmin=0, vmax=np.nanmax(H))
ax.set_xticks(range(len(cells))); ax.set_xticklabels(cells, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(present))); ax.set_yticklabels(present, fontsize=9)
for i in range(len(present)):
    for j in range(len(cells)):
        if np.isfinite(H[i, j]):
            ax.text(j, i, f"{H[i,j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if H[i, j] > 0.5*np.nanmax(H) else "black")
fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01, label="ECE")
ax.set_title("ECE by model x grid cell (darker = more miscalibrated)", fontsize=11)
fig.tight_layout(); fig.savefig(OUT_DIR / "ece_heatmap_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

# (b) calibration map: each point = one (model, cell); above diagonal = overconfident
fig, ax = plt.subplots(figsize=(6.2, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect calibration")
for m in present:
    xs, ys, ss = [], [], []
    for cl in cells:
        d = cal_all[(cal_all.model == m) & (cal_all.cell == cl)]
        if len(d): xs.append(d.correct.mean()); ys.append(d.confidence.mean()); ss.append(len(d))
    ax.scatter(xs, ys, s=[20 + 0.6*n for n in ss], color=MCOLOR[m], alpha=.7,
               edgecolor="white", linewidth=.5, label=m)
ax.set_xlabel("accuracy (per grid cell)"); ax.set_ylabel("mean confidence (per grid cell)")
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_aspect("equal"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title("Calibration map: each point = one (model, grid cell)\nabove diagonal = overconfident; size ∝ #items")
fig.tight_layout(); fig.savefig(OUT_DIR / "calibration_scatter.png", dpi=130, bbox_inches="tight"); plt.show()

# (c) confidence histograms per model
fig, axes = plt.subplots(1, len(present), figsize=(5*len(present), 3.6), sharey=True, squeeze=False)
for ax, m in zip(axes[0], present):
    ax.hist(cal_all[cal_all.model == m].confidence, bins=np.linspace(0, 1, 21), color=MCOLOR[m], alpha=.85)
    ax.set_title(f"{m}\nmean={cal_all[cal_all.model == m].confidence.mean():.2f}", fontsize=10)
    ax.set_xlabel("stated confidence"); ax.set_xlim(0, 1)
axes[0][0].set_ylabel("# questions")
fig.suptitle("Confidence distribution by model", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(OUT_DIR / "confidence_hist_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

## 9b. Cross-model comparison

Calibration of every model under test on the shared question set. qwen and gpt-5.4 score confidence from logprobs; claude estimates it by sampling (`AnthropicLLMPrompter`, K=`ANTHROPIC_N_SAMPLES`), so treat the logprob-vs-sampling difference as a method caveat. Only answers with a usable confidence value enter the calibration columns.

In [ ]:
# Overall calibration by model + ECE per (model x reasoning type).
cal_all = df_all[df_all["confidence"].notna()]
present = [m for m in MUT_MODELS if m in cal_all["model"].unique()]

cmp_rows = []
for m in present:
    d = cal_all[cal_all.model == m]
    cmp_rows.append({"model": m, "n": len(d), "accuracy": d.correct.mean(),
                     "mean_conf": d.confidence.mean(), "mean_entropy": d.norm_entropy.mean(),
                     "ece": m_ece(d), "gap": d.confidence.mean() - d.correct.mean()})
cross = pd.DataFrame(cmp_rows)
cross.to_csv(OUT_DIR / "metrics_by_model.csv", index=False)
print("Overall calibration by model (gap = mean confidence - accuracy):")
print(cross.round(3).to_string(index=False))

print("\nECE by model x reasoning type:")
ece_mt = pd.DataFrame(index=present, columns=list(GRID), dtype=float)
for m in present:
    for t in GRID:
        d = cal_all[(cal_all.model == m) & (cal_all.rtype == t)]
        ece_mt.loc[m, t] = m_ece(d) if len(d) else np.nan
print(ece_mt.round(3).to_string())

### Proper scoring rules vs ECE, weighted and unweighted

Per-model calibration under ECE and two strictly proper scores (Brier, NLL), each
computed unweighted and under the distribution weights (`*_w`). If Brier and NLL
induce the **same model ordering** as ECE, the ECE-based conclusion is not a binning
artefact. `ess` is the Kish effective sample size after weighting (n/ess = design
effect = variance inflation).

In [ ]:
# === ECE vs proper scores: per-model ranking, weighted & unweighted ==========
def _rank(s):                      # 1 = best (lowest error)
    return s.rank(method="min").astype(int)

_rows = []
for m in present:
    d = cal_all[cal_all.model == m]
    _rows.append({"model": m, "n": len(d), "ess": round(ess(d["weight"].values), 1),
                  "accuracy": m_accuracy(d), "mean_conf": m_meanconf(d),
                  "ece": m_ece(d), "brier": m_brier(d), "nll": m_nll(d),
                  "ece_w": m_ece(d, True), "brier_w": m_brier(d, True), "nll_w": m_nll(d, True)})
score = pd.DataFrame(_rows).set_index("model")
print("Per-model calibration (lower ece/brier/nll = better calibrated):")
print(score.round(4).to_string())

_metrics = ["ece", "brier", "nll", "ece_w", "brier_w", "nll_w"]
ranks = score[_metrics].apply(_rank)
print("\nModel ranking by each metric (1 = best calibrated):")
print(ranks.to_string())

ref = "ece"
print(f"\nDoes each metric induce the SAME ordering as unweighted ECE?")
for c in _metrics:
    if c == ref: continue
    same = bool((ranks[c] == ranks[ref]).all())
    print(f"  {c:8s}: {'AGREES ' if same else 'DIFFERS'}  order={list(score[c].sort_values().index)}")

score.to_csv(OUT_DIR / "metrics_by_model_scored.csv")
print("\nsaved", OUT_DIR / "metrics_by_model_scored.csv")


In [ ]:
# Reliability overlay + accuracy-vs-confidence bars across models.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]; ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect")
for m in present:
    d = cal_all[cal_all.model == m]
    pts = reliability_curve(d.confidence.values, d.correct.values)
    if pts:
        xs, ys, _ = zip(*pts)
        ax.plot(xs, ys, "-o", label=f"{m} (acc={d.correct.mean():.2f}, ECE={m_ece(d):.2f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.legend(fontsize=8); ax.set_title("Reliability by model")

ax = axes[1]
x = np.arange(len(present)); w = 0.38
ax.bar(x - w/2, [cal_all[cal_all.model == m].correct.mean() for m in present], w, label="accuracy")
ax.bar(x + w/2, [cal_all[cal_all.model == m].confidence.mean() for m in present], w, label="mean confidence")
ax.set_xticks(x); ax.set_xticklabels(present, rotation=20, ha="right", fontsize=8)
ax.set_ylim(0, 1); ax.legend(); ax.set_title("Accuracy vs mean confidence by model")
fig.tight_layout(); fig.savefig(OUT_DIR / "comparison_by_model.png", dpi=130, bbox_inches="tight")
plt.show()

### Rank-reversal check: this distribution vs the other

Loads the per-model scores saved by the *other* notebook
(`metrics_by_model_scored.csv`) and checks, for each metric, whether the model
ordering flips between the standard (MMLU-Pro) and target (REFLECT-failure)
distributions. A reversal that shows up under ECE **and** under the proper scores
(Brier/NLL) is robust; one that appears only under ECE is likely a binning artefact.
Run the scoring cell in *both* notebooks first.

In [ ]:
# === Rank-reversal check: this distribution vs the other =====================
from reflect.core.paths import analysis_experiment_dir
THIS_LABEL = "target (REFLECT failures)"
OTHER_DIR  = analysis_experiment_dir("llm_calibration_standard_entropy")
other_csv  = OTHER_DIR / "metrics_by_model_scored.csv"

if not other_csv.exists():
    print(f"Run the scoring cell in the OTHER notebook first to produce:\n  {other_csv}")
else:
    other = pd.read_csv(other_csv, index_col=0)
    common = [m for m in score.index if m in other.index]
    print(f"this = {THIS_LABEL}")
    print(f"common models: {common}")
    if len(common) < 2:
        print("Need >= 2 models present in BOTH notebooks to detect a reversal. "
              "(Tip: run the gpt-5.4 retry cell so it has usable data in both.)")
    else:
        any_rev = False
        for metric in ["ece", "ece_w", "brier", "brier_w", "nll", "nll_w"]:
            a = score.loc[common, metric].sort_values()
            b = other.loc[common, metric].sort_values()
            ra = {m: i for i, m in enumerate(a.index)}
            rb = {m: i for i, m in enumerate(b.index)}
            rev = [(x, y) for i, x in enumerate(common) for y in common[i + 1:]
                   if (ra[x] - ra[y]) * (rb[x] - rb[y]) < 0]
            any_rev = any_rev or bool(rev)
            print(f"[{metric:8s}] this={list(a.index)}  other={list(b.index)}  "
                  f"-> {'RANK REVERSAL ' + str(rev) if rev else 'same order'}")
        print("\nVERDICT:", "reversal present" if any_rev else "no reversal in the current data",
              "| a reversal is only credible if it holds under brier/nll too, not ECE alone.")


## 10. Save results

In [ ]:
# One row per question (with options + ground truth) for full reproducibility.
q_records = []
for q in questions:
    rec = {"qid": q.qid, "task": q.task, "episode": q.episode, "rtype": q.rtype,
           "regime": q.regime, "stem": q.stem, "context": q.context,
           "answer": q.answer, "correct_text": q.correct, "kinds": q.kinds,
           "options": q.options, "solvability": solvability[q.qid]}
    q_records.append(rec)
(OUT_DIR / "questions.jsonl").write_text("\n".join(json.dumps(r) for r in q_records))

df_all.to_csv(OUT_DIR / "results.csv", index=False)
cell_tbl.to_csv(OUT_DIR / "metrics_by_cell.csv", index=False)
by_type.to_csv(OUT_DIR / "metrics_by_type.csv", index=False)
by_regime.to_csv(OUT_DIR / "metrics_by_regime.csv", index=False)
by_diff.to_csv(OUT_DIR / "metrics_by_difficulty.csv", index=False)

print("saved to", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("  ", p.name)

## Composition sensitivity (does the easy-question skew drive the result?)

The generator emits more easy / saturated items than hard ones, so the count-weighted (micro) ECE
is diluted by trivially-calibrated cases. Here we **re-weight every item so each difficulty level
(panel solvability 0..5) contributes equally** - a composition-neutral view that corrects for the
generator's skew, using all the data (no sub-sampling). We also track ECE as easy items are
progressively dropped. Across all three views the model ordering is unchanged and the miscalibration
gaps grow, so the conclusions are not an artifact of the question mix - even if the engine is skewed
toward easier questions, the findings hold and strengthen.

In [ ]:
# Composition-neutral re-weighting: equalize each difficulty level (uses all data)
def balance_weights(d, by="solvability"):
    cnt = d[by].value_counts(); G = len(cnt); N = len(d)
    return d[by].map(lambda g: N / (G * cnt[g])).values

def weighted_ece(conf, correct, w, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float); w = np.asarray(w, float)
    edges = np.linspace(0, 1, n_bins + 1); W = w.sum(); tot = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mk = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        wb = w[mk].sum()
        if wb > 0:
            acc = (w[mk]*correct[mk]).sum()/wb; cf = (w[mk]*conf[mk]).sum()/wb
            tot += wb/W * abs(acc - cf)
    return tot

comp_rows = []
for m in present:
    d = cal_all[cal_all.model == m]; w = balance_weights(d)
    dn = cal_all[(cal_all.model == m) & (cal_all.rtype.isin(["T2", "T3"]))]
    comp_rows.append({"model": m,
        "micro_acc": d.correct.mean(), "balanced_acc": (w*d.correct.values).sum()/w.sum(),
        "micro_ece": ece(d.confidence.values, d.correct.values),
        "balanced_ece": weighted_ece(d.confidence.values, d.correct.values, w),
        "nontrivial_ece": ece(dn.confidence.values, dn.correct.values)})
comp = pd.DataFrame(comp_rows); comp.to_csv(OUT_DIR / "metrics_composition.csv", index=False)
print("Composition sensitivity (re-weighting equalizes panel-solvability levels 0..5):")
print(comp.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(present)); wbar = 0.26
ax.bar(x - wbar, comp.micro_ece,      wbar, label="micro (as generated)", color="#bbbbbb")
ax.bar(x,        comp.balanced_ece,   wbar, label="difficulty-balanced",  color="#4c78a8")
ax.bar(x + wbar, comp.nontrivial_ece, wbar, label="T2+T3 only (diagnostic)", color="#c44e52")
ax.set_xticks(x); ax.set_xticklabels(present); ax.set_ylabel("ECE"); ax.legend(fontsize=8)
ax.set_title("ECE by question composition (model ordering is invariant)")
for i, r in comp.iterrows():
    for off, val in [(-wbar, r.micro_ece), (0, r.balanced_ece), (wbar, r.nontrivial_ece)]:
        ax.text(i + off, val + 0.004, f"{val:.2f}", ha="center", fontsize=7)
fig.tight_layout(); fig.savefig(OUT_DIR / "composition_sensitivity_ece.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ECE as easy items are progressively dropped (keep only solvability <= cutoff)
thrs = [5, 4, 3, 2, 1, 0]
fig, ax = plt.subplots(figsize=(7.8, 4.4))
for m in present:
    d = cal_all[cal_all.model == m]
    ys = [ece(d[d.solvability <= t].confidence.values, d[d.solvability <= t].correct.values) for t in thrs]
    ax.plot(range(len(thrs)), ys, "-o", color=MCOLOR[m], label=m)
ns = [int((cal_all[cal_all.model == present[0]].solvability <= t).sum()) for t in thrs]
ax.set_xticks(range(len(thrs)))
ax.set_xticklabels([("all" if t == 5 else f"<= {t}") + f"\n(n={n})" for t, n in zip(thrs, ns)], fontsize=8)
ax.set_xlabel("items kept (max panel solvability)  -  easy items dropped left -> right")
ax.set_ylabel("ECE"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title("ECE as the set is restricted to harder items (max panel solvability = t)")
fig.tight_layout(); fig.savefig(OUT_DIR / "composition_ece_vs_threshold.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Reliability: as-generated vs difficulty-balanced (re-weighted)
def weighted_reliability(conf, correct, w, n_bins=ECE_BINS):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float); w = np.asarray(w, float)
    edges = np.linspace(0, 1, n_bins + 1); W = w.sum(); pts = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mk = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        wb = w[mk].sum()
        if wb > 0: pts.append(((w[mk]*conf[mk]).sum()/wb, (w[mk]*correct[mk]).sum()/wb, wb/W))
    return pts

fig, axes = plt.subplots(1, len(present), figsize=(4.6*len(present), 4.4), squeeze=False)
for ax, m in zip(axes[0], present):
    d = cal_all[cal_all.model == m]; w = balance_weights(d)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    up = reliability_curve(d.confidence.values, d.correct.values)
    if up:
        xs, ys, _ = zip(*up)
        ax.plot(xs, ys, "-o", color="#bbbbbb", ms=4, label=f"as-generated (ECE {ece(d.confidence.values, d.correct.values):.2f})")
    bp = weighted_reliability(d.confidence.values, d.correct.values, w)
    if bp:
        xs, ys, _ = zip(*bp)
        ax.plot(xs, ys, "-o", color=MCOLOR[m], label=f"balanced (ECE {weighted_ece(d.confidence.values, d.correct.values, w):.2f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.set_title(m, fontsize=10)
    ax.set_xlabel("mean confidence"); ax.legend(fontsize=7, loc="upper left")
axes[0][0].set_ylabel("accuracy")
fig.suptitle("Reliability: as-generated vs difficulty-balanced", y=1.02, fontsize=11)
fig.tight_layout(); fig.savefig(OUT_DIR / "reliability_balanced_by_model.png", dpi=130, bbox_inches="tight"); plt.show()

## Qualitative human validation (stratified sample)

A stratified random sample of ~100 questions (as equal as possible across the 13 type x regime
cells) for manual audit. For each item you mark whether it is **degenerate** and rate its
**difficulty** (easy / medium / hard) WITHOUT seeing the panel's rating. We then report the
degenerate rate and compare your difficulty ratings to the five-LLM panel's solvability.

In [ ]:
# Build the stratified sample (seeded, reproducible). Does NOT store panel difficulty,
#    so your difficulty rating is not biased by it. ────────────────────────────
SAMPLE_SIZE = 100
SAMPLE_CSV  = OUT_DIR / "human_eval_sample.csv"

_cells = [f"{t}x{r}" for t in GRID for r in GRID[t]]
_base, _rem = divmod(SAMPLE_SIZE, len(_cells))
_alloc = {cl: _base + (1 if k < _rem else 0) for k, cl in enumerate(_cells)}   # near-equal, sums to SAMPLE_SIZE
_by = defaultdict(list)
for q in questions: _by[f"{q.rtype}x{q.regime}"].append(q)
_rng = random.Random(f"{SEED}|human-sample")
_picked = []
for cl in _cells:
    pool = sorted(_by[cl], key=lambda q: q.qid); _rng.shuffle(pool)
    _picked.extend(pool[:min(_alloc[cl], len(pool))])
_rng.shuffle(_picked)                                   # randomize presentation order across cells

if SAMPLE_CSV.exists():
    sample_df = pd.read_csv(SAMPLE_CSV)
    print(f"{SAMPLE_CSV.name} already exists ({len(sample_df)} rows) - keeping your annotations. "
          f"Delete the file to resample.")
else:
    sample_df = pd.DataFrame([{
        "order": j, "qid": q.qid, "rtype": q.rtype, "regime": q.regime, "cell": f"{q.rtype}x{q.regime}",
        "stem": q.stem, "context": q.context,
        "optA": q.options["A"], "optB": q.options["B"], "optC": q.options["C"],
        "optD": q.options["D"], "optE": q.options["E"], "answer": q.answer, "correct_text": q.correct,
        "degenerate": "", "human_difficulty": ""}            # <- you fill these two columns
        for j, q in enumerate(_picked)])
    sample_df.to_csv(SAMPLE_CSV, index=False)
    print(f"wrote {SAMPLE_CSV.name}: {len(sample_df)} questions")

print("per-cell counts:", sample_df["cell"].value_counts().sort_index().to_dict())
print("annotate via the widget below, or edit 'degenerate' (0/1) and 'human_difficulty' "
      "(easy/medium/hard) directly in the CSV.")

In [ ]:
# Interactive annotator (writes to SAMPLE_CSV on every change; resumable)

_ann = pd.read_csv(SAMPLE_CSV)
for _c in ("degenerate", "human_difficulty"):
    if _c not in _ann.columns: _ann[_c] = ""
_ann["degenerate"] = _ann["degenerate"].astype("object")
_ann["human_difficulty"] = _ann["human_difficulty"].astype("object")
_st = {"i": 0}

def _deg_stored(v):
    try: return int(float(v))
    except (ValueError, TypeError): return ""
def _str(v): return "" if pd.isna(v) else str(v).strip()

_degW = widgets.ToggleButtons(options=[("- unset -", ""), ("not degenerate", 0), ("degenerate", 1)], description="Degenerate:")
_difW = widgets.ToggleButtons(options=[("- unset -", ""), ("easy", "easy"), ("medium", "medium"), ("hard", "hard")], description="Difficulty:")
_revW = widgets.ToggleButton(value=False, description="show answer key")
_qOut = widgets.Output(); _stat = widgets.HTML()
_prevB = widgets.Button(description="< Prev"); _nextB = widgets.Button(description="Next >")

def _n_done():
    return int(((_ann["degenerate"].map(_str) != "") & (_ann["human_difficulty"].map(_str) != "")).sum())
def _refresh_status():
    _stat.value = f"<b>{_n_done()}/{len(_ann)}</b> fully annotated &middot; autosaved to {SAMPLE_CSV.name}"
def _save():
    _ann.to_csv(SAMPLE_CSV, index=False)

def _render():
    r = _ann.iloc[_st["i"]]
    with _qOut:
        clear_output()
        opts = "\n".join(f"- **{L})** {r['opt'+L]}" + ("  &larr; correct" if (_revW.value and L == r["answer"]) else "")
                         for L in "ABCDE")
        display(Markdown(f"**{_st['i']+1} / {len(_ann)}**  &middot;  cell `{r['cell']}`  &middot;  `{r['qid']}`\n\n"
                         f"**Context:** {r['context']}\n\n**Question:** {r['stem']}\n\n{opts}"))
    dv = _deg_stored(r["degenerate"])
    _degW.unobserve(_on_deg, "value"); _degW.value = dv if dv in (0, 1) else ""; _degW.observe(_on_deg, "value")
    hv = _str(r["human_difficulty"])
    _difW.unobserve(_on_dif, "value"); _difW.value = hv if hv in ("easy", "medium", "hard") else ""; _difW.observe(_on_dif, "value")
    _refresh_status()

def _on_deg(ch): _ann.at[_st["i"], "degenerate"] = ch["new"]; _save(); _refresh_status()
def _on_dif(ch): _ann.at[_st["i"], "human_difficulty"] = ch["new"]; _save(); _refresh_status()
def _go(d): _st["i"] = (_st["i"] + d) % len(_ann); _render()

_prevB.on_click(lambda b: _go(-1)); _nextB.on_click(lambda b: _go(1))
_revW.observe(lambda ch: _render(), "value")
_degW.observe(_on_deg, "value"); _difW.observe(_on_dif, "value")
display(widgets.VBox([widgets.HBox([_prevB, _nextB, _revW]), _qOut, _degW, _difW, _stat]))
_render()

In [ ]:
# Analyze the human annotations: degenerate rate + human-vs-panel difficulty

ann = pd.read_csv(SAMPLE_CSV)
ann["degenerate"] = pd.to_numeric(ann["degenerate"], errors="coerce")
ann["human_difficulty"] = ann["human_difficulty"].astype(str).str.strip().str.lower()
done = ann[ann.degenerate.isin([0, 1]) & ann.human_difficulty.isin(["easy", "medium", "hard"])].copy()
print(f"annotated: {len(done)}/{len(ann)}")
if len(done) == 0:
    print("No annotations yet - use the widget above (or fill the CSV) first.")
else:
    # (1) DEGENERATE RATE
    print(f"\n[1] degenerate: {int(done.degenerate.sum())}/{len(done)} = {done.degenerate.mean()*100:.1f}%")
    bycell = done.groupby("cell").degenerate.agg(n="count", deg="sum")
    bycell["rate"] = (bycell.deg / bycell.n).round(2)
    print(bycell.to_string())

    # (2) HUMAN vs PANEL DIFFICULTY
    order = ["easy", "medium", "hard"]; omap = {"easy": 0, "medium": 1, "hard": 2}
    done["panel_solv"] = done.qid.map(solvability)
    done["panel_diff"] = done.panel_solv.map(difficulty_bucket)
    agree = (done.human_difficulty == done.panel_diff).mean()
    def _kappa(a, b, labels):
        cm = pd.crosstab(pd.Categorical(a, labels), pd.Categorical(b, labels), dropna=False).reindex(
            index=labels, columns=labels, fill_value=0).values.astype(float)
        n = cm.sum(); po = np.trace(cm) / n
        pe = sum((cm[i, :].sum() / n) * (cm[:, i].sum() / n) for i in range(len(labels)))
        return (po - pe) / (1 - pe) if pe < 1 else 1.0
    kappa = _kappa(done.human_difficulty, done.panel_diff, order)
    rho, pval = spearmanr(done.human_difficulty.map(omap), done.panel_solv)   # higher solvability = easier
    print(f"\n[2] human vs panel-bucket: exact agreement = {agree*100:.1f}% | Cohen's kappa = {kappa:.2f}")
    print(f"    Spearman(human-difficulty ordinal, panel solvability) = {rho:.2f} (p={pval:.3f}); "
          f"negative is expected (more panel-solvable = easier)")
    conf = pd.crosstab(pd.Categorical(done.human_difficulty, order),
                       pd.Categorical(done.panel_diff, order), dropna=False).reindex(index=order, columns=order, fill_value=0)
    print("\nconfusion (rows = human, cols = panel):"); print(conf.to_string())

    # plots: degenerate-by-cell bar + difficulty confusion heatmap
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    bycell["rate"].reindex([f"{t}x{r}" for t in GRID for r in GRID[t]]).plot.bar(ax=a1, color="#d62728")
    a1.set_title(f"Degenerate rate by cell (overall {done.degenerate.mean()*100:.0f}%)"); a1.set_ylabel("fraction degenerate"); a1.set_ylim(0, 1)
    im = a2.imshow(conf.values, cmap="Blues")
    a2.set_xticks(range(3)); a2.set_xticklabels(order); a2.set_yticks(range(3)); a2.set_yticklabels(order)
    a2.set_xlabel("panel difficulty"); a2.set_ylabel("human difficulty")
    for i in range(3):
        for j in range(3): a2.text(j, i, int(conf.values[i, j]), ha="center", va="center",
                                   color="white" if conf.values[i, j] > conf.values.max()/2 else "black")
    a2.set_title(f"Difficulty agreement (acc {agree*100:.0f}%, kappa {kappa:.2f})")
    fig.tight_layout(); fig.savefig(OUT_DIR / "human_eval_summary.png", dpi=130, bbox_inches="tight"); plt.show()

## 11. Reviewer checks: renormalization worked examples (#1) and gpt-5.4 reliability, generic vs deployment (#5)

**Check #1 - does the option count mechanically inflate confidence?** No. The cell below pulls the
*cached* renormalized option distribution for a gpt-5.4 item from each dataset (5-option target,
10-option MMLU-Pro) and shows the entropy -> `confidence = 1 - H/log(K)` derivation. Because the
normaliser is `log(#options)`, the option count is divided out: gpt-5.4 returns a near one-hot
letter distribution on *both* the 10-option and the 5-option items, so its ~0.94 mean confidence is
its own near-deterministic output, not a 5-option artefact. (`rel.logit` = `log p_i - log p_top`,
the raw next-token logit gap among letters; the renormalization constant cancels.)

**Check #5 - how does calibration degrade?** The side-by-side reliability diagram shows gpt-5.4 is
not uniformly overconfident: on the target it *saturates at the top bin* - the large majority of
items sit at confidence ~ 1.0 where accuracy is only ~0.73, with a non-monotonic dip below.

In [ ]:
# --- Reviewer checks #1 and #5: renormalization worked examples + gpt-5.4 reliability side-by-side ---
# Self-contained: reads the two notebooks' saved caches/results from disk. No LLM re-query.
import json, math, glob, hashlib, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

STD_DIR = OUT_DIR.parent / "llm_calibration_standard_entropy"   # sibling standard (MMLU-Pro) run
TGT_DIR = OUT_DIR

def _norm_entropy(probs):
    p = np.asarray(list(probs.values()), float); K = p.size
    p = p[p > 0]
    H = float(-np.sum(p * np.log(p)))            # Shannon entropy in nats
    return H, H / math.log(K), K                 # H, normalized entropy, #options

def _key(model, ans_sys, user):                  # mirrors _cache_key in both notebooks
    return hashlib.sha256(f"{model}\x00{ans_sys}\x00{user}".encode()).hexdigest()

def _show(tag, header, options, answer, predicted, probs, correct):
    H, nH, K = _norm_entropy(probs); conf = 1 - nH
    print(f"\n{'='*78}\n{tag}\n{'='*78}\n{header}")
    print(f"answer={answer}  predicted={predicted}  correct={bool(correct)}   (#options K={K})")
    top = max(probs.values())
    print(f"{'opt':>3} {'renorm p':>10} {'rel.logit':>10}  option text")
    for L, p in sorted(probs.items(), key=lambda kv: -kv[1]):
        rel = math.log(p) - math.log(top) if p > 0 else float('-inf')   # raw letter-logit minus top (offset cancels in renorm)
        print(f"{L:>3} {p:10.5f} {rel:10.2f}  {str(options[L])[:52]}")
    print(f"  Shannon H = {H:.4f} nats ;  normaliser log(K) = log({K}) = {math.log(K):.4f}")
    print(f"  norm_entropy = H/log(K) = {nH:.4f}   ->   confidence = 1 - norm_entropy = {conf:.4f}")

# ---- Worked examples (Check #1): the cached `option_probs` are the renormalized over-letters
#      distribution; rel.logit = log p_i - log p_top equals the raw next-token logit gap among
#      letters (the renormalization constant cancels). confidence = 1 - H/log(K) divides out the
#      option count, so 5 vs 10 options cannot mechanically inflate confidence. ----
try:
    # TARGET (K=5): reconstruct the exact prompt from questions.jsonl, look up the cached gpt-5.4 entry
    SYS_T = ("You answer multiple-choice questions about a robot's task execution. "
             "Read the context and output ONLY the single letter (A-E) of the best option. "
             "No words, no punctuation, no explanation.")
    def _user_T(r):
        opts = "\n".join(f"{L} = {r['options'][L]}" for L in "ABCDE")
        return f"Context: {r['context']}\n\nQuestion: {r['stem']}\n{opts}\nAnswer:"
    qT = {json.loads(l)["qid"]: json.loads(l) for l in open(TGT_DIR / "questions.jsonl")}
    cT = json.load(open(TGT_DIR / "llm_cache.json"))
    dT = pd.read_csv(TGT_DIR / "results_all_models.csv"); dT["qid"] = dT["qid"].astype(str)
    gT = dT[dT.model == "gpt-5.4"]
    rowT = gT.sort_values("confidence", ascending=False).iloc[0]      # a saturated gpt example
    rec = qT[rowT.qid]; eT = cT[_key("gpt-5.4", SYS_T, _user_T(rec))]
    _show("TARGET (REFLECT failures, K=5) - gpt-5.4",
          f"qid={rowT.qid}  rtype={rec['rtype']}\nstem: {rec['stem'][:120]}",
          rec["options"], rec["answer"], eT["predicted_label"], eT["option_probs"], int(rowT.correct))

    # STANDARD (K=10): read MMLU-Pro from the local parquet cache (avoids the datasets lib), rebuild prompt
    SYS_S = ("You answer multiple-choice questions. Read the question and output ONLY the single "
             "letter of the best option. No words, no punctuation, no explanation.")
    LET = "ABCDEFGHIJ"
    def _user_S(stem, options, letters):
        return f"Question: {stem}\n" + "\n".join(f"{L} = {options[L]}" for L in letters) + "\nAnswer:"
    pq = glob.glob(str(pathlib.Path.home() /
          ".cache/huggingface/hub/datasets--TIGER-Lab--MMLU-Pro/snapshots/*/data/*.parquet"))
    mp = pd.concat([pd.read_parquet(p) for p in pq], ignore_index=True)
    dmap = {int(r.question_id): r for r in mp.itertuples(index=False)}
    cS = json.load(open(STD_DIR / "llm_cache.json"))
    dS = pd.read_csv(STD_DIR / "results_all_models.csv")
    gS = dS[(dS.model == "gpt-5.4") & (dS.n_options == 10)]
    for tag, rowS in [("STANDARD (MMLU-Pro, K=10) - gpt-5.4 [near-deterministic, like the target]",
                       gS.sort_values("confidence", ascending=False).iloc[0]),
                      ("STANDARD (MMLU-Pro, K=10) - gpt-5.4 [a genuinely uncertain item, for contrast]",
                       gS.sort_values("confidence").iloc[0])]:
        r = dmap[int(rowS.qid)]; opts = list(r.options); letters = list(LET[:len(opts)])
        options = {L: str(opts[i]) for i, L in enumerate(letters)}
        eS = cS[_key("gpt-5.4", SYS_S, _user_S(r.question, options, letters))]
        _show(tag, f"qid={int(rowS.qid)}  category={r.category}\nstem: {r.question[:120]}",
              options, r.answer, eS["predicted_label"], eS["option_probs"], int(rowS.correct))
    print("\nTakeaway: gpt-5.4 returns a near one-hot letter distribution on BOTH the 10-option generic "
          "items and the 5-option target items, so its high confidence is its own near-deterministic "
          "output, not an artefact of the option count (which log(K) normalises out).")
except Exception as e:
    print(f"[worked-example reconstruction skipped: {type(e).__name__}: {e}]")

# ---- Reliability diagrams side by side (Check #5): gpt-5.4 generic vs deployment ----
NB = 10
def _ece(conf, corr, nb=NB):
    conf = np.asarray(conf, float); corr = np.asarray(corr, float); edges = np.linspace(0, 1, nb + 1); t = 0.0
    for b in range(nb):
        m = (conf > edges[b]) & (conf <= edges[b+1]) if b > 0 else (conf >= edges[b]) & (conf <= edges[b+1])
        if m.sum(): t += m.sum() / len(conf) * abs(corr[m].mean() - conf[m].mean())
    return t
def _curve(conf, corr, nb=NB):
    conf = np.asarray(conf, float); corr = np.asarray(corr, float); edges = np.linspace(0, 1, nb + 1); pts = []
    for b in range(nb):
        m = (conf > edges[b]) & (conf <= edges[b+1]) if b > 0 else (conf >= edges[b]) & (conf <= edges[b+1])
        if m.sum(): pts.append((conf[m].mean(), corr[m].mean(), int(m.sum())))
    return pts

panels = [("Standard (MMLU-Pro)", STD_DIR / "results_all_models.csv"),
          ("Target (REFLECT failures)", TGT_DIR / "results_all_models.csv")]
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharex=True, sharey=True)
for ax, (name, path) in zip(axes, panels):
    d = pd.read_csv(path); d = d[(d.model == "gpt-5.4") & d.confidence.notna()]
    pts = _curve(d.confidence, d.correct)
    xs = [a for a, _, _ in pts]; ys = [b for _, b, _ in pts]; ns = [n for *_, n in pts]
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    ax.scatter(xs, ys, s=[20 + 12 * n ** 0.5 for n in ns], color="#d62728", alpha=.8, zorder=3)
    ax.plot(xs, ys, color="#d62728", lw=1.2, zorder=2)
    for x, y, n in pts:
        ax.annotate(str(n), (x, y), fontsize=7, xytext=(2, 3), textcoords="offset points")
    ax.set_title(f"gpt-5.4 - {name}\nacc={d.correct.mean():.2f}  mean_conf={d.confidence.mean():.2f}  "
                 f"ECE={_ece(d.confidence, d.correct):.3f}", fontsize=11)
    ax.set_xlabel("mean predicted confidence"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect("equal"); ax.grid(alpha=.25)
axes[0].set_ylabel("empirical accuracy")
fig.suptitle("Reliability: gpt-5.4, generic benchmark vs deployment distribution "
             "(points labelled with bin count)", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(TGT_DIR / "reliability_gpt_standard_vs_target.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved", TGT_DIR / "reliability_gpt_standard_vs_target.png")
